# Lesson 13B: Diffusion Models - Practical

## Training a Denoising Diffusion Model

**Case Study**: Train a simplified diffusion model to generate 2D data. Learn how iterative denoising creates high-quality samples.

**Key Idea**: Learn to reverse the noise addition process.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons

torch.manual_seed(42)
np.random.seed(42)
print('✅ Loaded!')

## Step 1: Define Diffusion Schedule

**Forward process**: Gradually add Gaussian noise  
**β schedule**: Controls noise level at each timestep

In [ ]:
def get_beta_schedule(T=100, beta_start=0.0001, beta_end=0.02):
    """Linear beta schedule"""
    return torch.linspace(beta_start, beta_end, T)

def get_alpha_schedule(betas):
    """Compute alpha and cumulative alpha"""
    alphas = 1 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    return alphas, alphas_cumprod

# Create schedule
T = 100
betas = get_beta_schedule(T)
alphas, alphas_cumprod = get_alpha_schedule(betas)

print(f"Timesteps: {T}")
print(f"Beta range: [{betas[0]:.4f}, {betas[-1]:.4f}]")
print(f"Alpha_bar range: [{alphas_cumprod[0]:.4f}, {alphas_cumprod[-1]:.4f}]")

# Visualize schedule
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(betas.numpy(), linewidth=2)
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('β_t', fontsize=11)
ax.set_title('Noise Schedule (β)', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(alphas_cumprod.numpy(), linewidth=2, color='red')
ax.set_xlabel('Timestep t', fontsize=11)
ax.set_ylabel('ᾱ_t', fontsize=11)
ax.set_title('Cumulative Alpha (signal retention)', fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Diffusion schedule created!')

## Step 2: Forward Diffusion Process

q(x_t | x_0) = N(x_t; √ᾱ_t x_0, (1-ᾱ_t)I)

This allows sampling x_t directly from x_0 without iterating.

In [ ]:
def forward_diffusion(x0, t, alphas_cumprod):
    """Add noise to x0 according to timestep t"""
    batch_size = x0.shape[0]
    
    # Get alpha_bar for each sample's timestep
    sqrt_alpha_bar = torch.sqrt(alphas_cumprod[t])
    sqrt_one_minus_alpha_bar = torch.sqrt(1 - alphas_cumprod[t])
    
    # Sample noise
    noise = torch.randn_like(x0)
    
    # Forward diffusion: x_t = √ᾱ_t * x_0 + √(1-ᾱ_t) * ε
    x_t = sqrt_alpha_bar.view(-1, 1) * x0 + sqrt_one_minus_alpha_bar.view(-1, 1) * noise
    
    return x_t, noise

# Demonstrate forward process
X_real, _ = make_moons(n_samples=500, noise=0.05, random_state=42)
X_real = torch.FloatTensor(X_real)

# Show noise addition at different timesteps
timesteps = [0, 25, 50, 75, 99]
fig, axes = plt.subplots(1, len(timesteps), figsize=(16, 3))

for i, t in enumerate(timesteps):
    t_tensor = torch.full((len(X_real),), t, dtype=torch.long)
    X_noisy, _ = forward_diffusion(X_real, t_tensor, alphas_cumprod)
    
    ax = axes[i]
    ax.scatter(X_noisy[:, 0], X_noisy[:, 1], s=10, alpha=0.5)
    ax.set_title(f't = {t}', fontweight='bold')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.grid(True, alpha=0.3)

plt.suptitle('Forward Diffusion Process', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('✅ Forward process demonstrated!')

## Step 3: Define Denoising Network

The network predicts the noise ε added at each timestep.

In [ ]:
class SimpleDiffusion(nn.Module):
    def __init__(self, data_dim=2, time_dim=32, hidden_dim=128):
        super().__init__()
        # Time embedding
        self.time_embed = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        
        # Denoising network
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, data_dim)
        )
    
    def forward(self, x, t):
        # Embed time
        t_normalized = t.unsqueeze(-1).float() / 100.0
        t_emb = self.time_embed(t_normalized)
        
        # Concatenate and denoise
        x_t = torch.cat([x, t_emb], dim=-1)
        return self.net(x_t)

model = SimpleDiffusion(data_dim=2, time_dim=32, hidden_dim=128)
print('✅ Denoising network defined')
print(f'   Parameters: {sum(p.numel() for p in model.parameters())}')

## Step 4: Train Denoising Model

**Loss**: MSE between predicted noise and true noise

In [ ]:
# Training setup
optimizer = optim.Adam(model.parameters(), lr=0.001)
n_epochs = 500
batch_size = 128

# Prepare full dataset
X_train, _ = make_moons(n_samples=2000, noise=0.05, random_state=42)
X_train = torch.FloatTensor(X_train)

losses = []

print("Training diffusion model...")
for epoch in range(n_epochs):
    # Sample batch
    idx = torch.randint(0, len(X_train), (batch_size,))
    batch = X_train[idx]
    
    # Sample random timesteps
    t = torch.randint(0, T, (batch_size,))
    
    # Forward diffusion
    x_noisy, noise_true = forward_diffusion(batch, t, alphas_cumprod)
    
    # Predict noise
    noise_pred = model(x_noisy, t)
    
    # Compute loss
    loss = nn.MSELoss()(noise_pred, noise_true)
    
    # Backprop
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f'Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.4f}')

print('✅ Diffusion model trained!')

## Step 5: Sampling (Reverse Diffusion)

Start from pure noise and iteratively denoise.

In [ ]:
@torch.no_grad()
def sample(model, n_samples, T, betas, alphas, alphas_cumprod):
    """Generate samples using reverse diffusion"""
    model.eval()
    
    # Start from pure noise
    x = torch.randn(n_samples, 2)
    
    # Reverse diffusion
    for t in reversed(range(T)):
        t_tensor = torch.full((n_samples,), t, dtype=torch.long)
        
        # Predict noise
        noise_pred = model(x, t_tensor)
        
        # Compute x_{t-1}
        alpha_t = alphas[t]
        alpha_bar_t = alphas_cumprod[t]
        beta_t = betas[t]
        
        # Mean of reverse distribution
        mean = (1 / torch.sqrt(alpha_t)) * (x - (beta_t / torch.sqrt(1 - alpha_bar_t)) * noise_pred)
        
        if t > 0:
            noise = torch.randn_like(x)
            x = mean + torch.sqrt(beta_t) * noise
        else:
            x = mean
    
    return x

# Generate samples
n_samples = 500
samples = sample(model, n_samples, T, betas, alphas, alphas_cumprod)

print(f'✅ Generated {n_samples} samples!')

## Step 6: Visualize Results

In [ ]:
# Compare real vs generated
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Real data
ax = axes[0]
ax.scatter(X_train[:500, 0], X_train[:500, 1], s=20, alpha=0.6, c='blue')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Real Data', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Generated data
ax = axes[1]
ax.scatter(samples[:, 0], samples[:, 1], s=20, alpha=0.6, c='red')
ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('Generated (Diffusion)', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Training loss
ax = axes[2]
ax.plot(losses, linewidth=1, alpha=0.7)
ax.set_xlabel('Iteration', fontsize=11)
ax.set_ylabel('MSE Loss', fontsize=11)
ax.set_title('Training Loss', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('✅ Diffusion model successfully learned the distribution!')

## Summary

**Diffusion Model Achievements**:
- ✅ Learned to denoise at each timestep
- ✅ Generated high-quality samples from pure noise
- ✅ Stable training (no mode collapse like GANs)

**Key Advantages**:
1. **Stability**: Much more stable than GANs
2. **Quality**: State-of-the-art image generation
3. **Likelihood**: Can compute log-likelihood
4. **Inpainting**: Easy to do conditional generation

**Production Tips**:
- Use U-Net architecture for images
- Add classifier-free guidance for conditional generation
- Use DDIM sampling for faster generation (fewer steps)
- Applications: DALL-E 2, Stable Diffusion, Imagen

**Diffusion vs GAN vs VAE**:
- Diffusion: Best quality, slow sampling, stable training
- GAN: Fast sampling, training unstable, mode collapse
- VAE: Fast, blurry samples, explicit likelihood